# Naming Conventions

Naming is not cosmetic. In real codebases, names determine how quickly someone can reason about state, ownership, side effects, and intent. Bad naming creates hidden latency in every review, incident, and refactor.


## 1. Problem First

Why does this matter in real systems?

- A variable called `data` hides whether it is raw input, normalized output, or cached state.
- A function named `handle()` tells nothing about side effects.
- Ambiguous names multiply mistakes in large teams and long-lived services.


In [3]:
raw_log_line = "2026-05-01|ERROR|db timeout"
parsed_fields = raw_log_line.split("|")
severity = parsed_fields[1]

print(severity)

ERROR


## 2. Minimal Concept

Core syntax and conventions:

- Use `snake_case` for variables and functions.
- Use `PascalCase` for classes.
- Use `UPPER_CASE` for constants.
- Prefer names that reveal role, units, and lifecycle stage.


## 3. Mental Model

How Python actually behaves

Python does not enforce semantic meaning in names, but humans do. Names are part of the program's operational interface for maintainers. A good name compresses context. A bad name forces readers to rebuild context from surrounding code every time.


In [6]:
MAX_RETRY_SECONDS = 30

class LogParser:
    pass

def parse_log_line(line):
    return line.split("|")

print(MAX_RETRY_SECONDS)
print(LogParser.__name__)
print(parse_log_line("a|b|c"))

30
LogParser
['a', 'b', 'c']


## 4. Internal Mechanics

Names are keys in namespaces. Their style does not change runtime semantics, but it changes how people infer meaning. Tools also rely on conventions: linters, autocomplete, frameworks, and readers all extract signal from consistent naming.


In [8]:
module_namespace = {}
exec("request_timeout_seconds = 15", module_namespace)
print(module_namespace["request_timeout_seconds"])
print("request_timeout_seconds" in module_namespace)

15
True


## 5. Edge Cases

Where it breaks:

- `l`, `O`, and `I` are visually dangerous names.
- `data`, `obj`, `temp`, and `value` often hide domain meaning.
- Reusing one name for multiple lifecycle stages causes state confusion.
- Overly long names can be as harmful as vague names if they bury the main idea.


In [10]:
l = [1, 2, 3]
O = 0
I = 1

print(l, O, I)

normalized_user_payload = {"user_id": 7, "status": "active"}
print(normalized_user_payload)

[1, 2, 3] 0 1
{'user_id': 7, 'status': 'active'}


## 6. Performance Thinking

Naming has negligible runtime cost, but strong engineering impact:

- Good names reduce debugging and onboarding time.
- Clear unit suffixes like `_ms`, `_seconds`, `_bytes` prevent expensive production mistakes.
- Consistent naming lowers search friction across large codebases.


## 7. Real Use Case

In a CLI or pipeline, naming the same payload differently across stages creates bugs because engineers stop knowing whether they are handling raw, validated, or transformed data.


In [13]:
raw_payload = {"user_id": "42", "active": "true"}
validated_payload = {"user_id": 42, "active": True}
serialized_payload = '{"user_id": 42, "active": true}'

print(raw_payload)
print(validated_payload)
print(serialized_payload)

{'user_id': '42', 'active': 'true'}
{'user_id': 42, 'active': True}
{"user_id": 42, "active": true}


## 8. Anti-Pattern

What beginners do wrong:

- Use generic names because they are quick to type.
- Rename things based on implementation detail instead of business role.
- Use one variable like `data` for five transformations in a row.


In [15]:
# Hard to reason about
data = "42,true"
data = data.split(",")
data = {"user_id": int(data[0]), "active": data[1] == "true"}
print(data)

# Easier to audit
raw_csv_record = "42,true"
fields = raw_csv_record.split(",")
user_record = {"user_id": int(fields[0]), "active": fields[1] == "true"}
print(user_record)

{'user_id': 42, 'active': True}
{'user_id': 42, 'active': True}


## 9. Interview Signals

Questions FAANG asks:

- What makes a name good in a large production codebase?
- Why are unit-bearing names important?
- When is a short name acceptable?
- How do naming conventions help tooling and maintenance?


## 10. Exercise (Non-trivial)

Refactor a small ETL snippet where every intermediate value is named `data`. Rename it so lifecycle, validation state, and serialization boundary are obvious. Then explain which future bug becomes easier to spot.


In [18]:
def run(data):
    data = data.strip()
    data = data.split(",")
    data = {"user_id": int(data[0]), "plan": data[1]}
    return data

# Task:
# 1. Rename each stage precisely.
# 2. Add one variable name that exposes units or format.
# 3. Explain why the refactor helps debugging.